In [7]:
import pandas as pd
from pathlib import Path
from collections import Counter

In [8]:
DATA_DIR = Path("../training_datasets")

S1_PATH = DATA_DIR / "train_source1.tsv"
S2_PATH = DATA_DIR / "train_source2.tsv"
S3_PATH = DATA_DIR / "train_source3.tsv"
GT_PATH = DATA_DIR / "train_ground_truth.tsv"

CHUNK_SIZE = 100_000

In [9]:
def profile_source(path, chunk_size=100_000):
    total_rows = 0

    missing_counts = Counter()
    country_counts = Counter()

    unique_ids = set()
    duplicate_ids = 0

    name_lengths = []
    address_lengths = []

    unique_names = set()
    unique_addresses = set()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        chunksize=chunk_size
    ):
        total_rows += len(chunk)

        # Missing values
        missing_counts.update(
            chunk.isna().sum().to_dict()
        )

        # Countries
        country_counts.update(
            chunk["country"].fillna("<MISSING>").value_counts().to_dict()
        )

        # Entity IDs
        ids = chunk["entity_id"].dropna()

        duplicate_ids += ids.duplicated().sum()

        # Check duplicates across chunks too
        for entity_id in ids:
            if entity_id in unique_ids:
                duplicate_ids += 1
            else:
                unique_ids.add(entity_id)

        # Name statistics
        names = chunk["business_name"].fillna("")

        name_lengths.extend(names.str.len().tolist())
        unique_names.update(names[names != ""])

        # Address statistics
        addresses = chunk["business_address"].fillna("")

        address_lengths.extend(addresses.str.len().tolist())
        unique_addresses.update(addresses[addresses != ""])

    print(f"Rows: {total_rows:,}")
    print(f"Columns: 4")

    print("\nMissing values:")
    for column, count in missing_counts.items():
        print(f"  {column}: {count:,}")

    print("\nEntity IDs:")
    print(f"  Unique IDs: {len(unique_ids):,}")
    print(f"  Duplicate IDs: {duplicate_ids:,}")

    print("\nCountries:")
    for country, count in country_counts.most_common():
        print(f"  {country}: {count:,}")

    print("\nBusiness name:")
    print(f"  Unique names: {len(unique_names):,}")
    print(f"  Average length: {sum(name_lengths) / len(name_lengths):.2f}")
    print(f"  Min length: {min(name_lengths)}")
    print(f"  Max length: {max(name_lengths)}")

    print("\nBusiness address:")
    print(f"  Unique addresses: {len(unique_addresses):,}")
    print(f"  Average length: {sum(address_lengths) / len(address_lengths):.2f}")
    print(f"  Min length: {min(address_lengths)}")
    print(f"  Max length: {max(address_lengths)}")

    return {
        "rows": total_rows,
        "unique_ids": len(unique_ids),
        "duplicate_ids": duplicate_ids,
        "country_counts": country_counts,
        "missing_counts": missing_counts,
    }

In [10]:
s1_profile = profile_source(S1_PATH)

Rows: 2,206,821
Columns: 4

Missing values:
  entity_id: 0
  business_name: 0
  business_address: 0
  country: 0

Entity IDs:
  Unique IDs: 2,206,821
  Duplicate IDs: 0

Countries:
  US: 1,323,633
  India: 883,188

Business name:
  Unique names: 1,539,229
  Average length: 24.03
  Min length: 3
  Max length: 105

Business address:
  Unique addresses: 2,130,606
  Average length: 52.07
  Min length: 11
  Max length: 256


In [11]:
s2_profile = profile_source(S2_PATH)

Rows: 5,034,616
Columns: 4

Missing values:
  entity_id: 0
  business_name: 2
  business_address: 168,967
  country: 0

Entity IDs:
  Unique IDs: 5,034,616
  Duplicate IDs: 0

Countries:
  US: 3,016,817
  India: 2,017,799

Business name:
  Unique names: 4,402,008
  Average length: 25.10
  Min length: 0
  Max length: 104

Business address:
  Unique addresses: 4,337,261
  Average length: 46.23
  Min length: 0
  Max length: 249


In [15]:
s3_profile = profile_source(S3_PATH)

Rows: 5,285,603
Columns: 4

Missing values:
  entity_id: 0
  business_name: 13
  business_address: 175,916
  country: 0

Entity IDs:
  Unique IDs: 5,285,603
  Duplicate IDs: 0

Countries:
  US: 3,170,056
  India: 2,115,547

Business name:
  Unique names: 4,651,608
  Average length: 25.20
  Min length: 0
  Max length: 123

Business address:
  Unique addresses: 4,632,764
  Average length: 46.71
  Min length: 0
  Max length: 240


# Profiling Ground Truth Function

In [13]:
def get_entity_ids(path, chunk_size=100_000):
    ids = set()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        usecols=["entity_id"],
        chunksize=chunk_size
    ):
        ids.update(chunk["entity_id"].dropna())

    return ids

In [14]:
s2_ids = get_entity_ids(S2_PATH)
s3_ids = get_entity_ids(S3_PATH)

print(f"S2 IDs: {len(s2_ids):,}")
print(f"S3 IDs: {len(s3_ids):,}")

S2 IDs: 5,034,616
S3 IDs: 5,285,603


In [16]:
def profile_ground_truth(path, s2_ids, s3_ids):
    total_rows = 0

    singleton_count = 0
    match_count_distribution = Counter()

    s2_match_count = 0
    s3_match_count = 0
    unknown_match_count = 0

    source1_ids = set()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        chunksize=100_000
    ):
        total_rows += len(chunk)

        # Source 1 IDs
        source1_ids.update(
            chunk["source1_entity_id"].dropna()
        )

        for matched_ids in chunk["matched_entity_ids"].fillna(""):
            matched_ids = matched_ids.strip()

            # Singleton
            if not matched_ids:
                singleton_count += 1
                match_count_distribution[0] += 1
                continue

            ids = [
                entity_id.strip()
                for entity_id in matched_ids.split(",")
                if entity_id.strip()
            ]

            match_count_distribution[len(ids)] += 1

            for entity_id in ids:
                if entity_id in s2_ids:
                    s2_match_count += 1

                elif entity_id in s3_ids:
                    s3_match_count += 1

                else:
                    unknown_match_count += 1

    print(f"Rows: {total_rows:,}")

    print("\nUnique Source 1 IDs:")
    print(f"  {len(source1_ids):,}")

    print("\nMatch count distribution:")

    for count, frequency in sorted(match_count_distribution.items()):
        print(f"  {count} matches: {frequency:,}")

    print("\nSingletons:")
    print(f"  {singleton_count:,}")
    print(f"  Singleton rate: {singleton_count / total_rows:.2%}")

    print("\nMatches by source:")
    print(f"  S2: {s2_match_count:,}")
    print(f"  S3: {s3_match_count:,}")
    print(f"  Unknown IDs: {unknown_match_count:,}")

    return {
        "rows": total_rows,
        "unique_source1_ids": len(source1_ids),
        "singleton_count": singleton_count,
        "singleton_rate": singleton_count / total_rows,
        "match_distribution": match_count_distribution,
        "s2_matches": s2_match_count,
        "s3_matches": s3_match_count,
        "unknown_matches": unknown_match_count,
    }

In [17]:
gt_profile = profile_ground_truth(
    GT_PATH,
    s2_ids,
    s3_ids
)

Rows: 2,206,821

Unique Source 1 IDs:
  2,206,821

Match count distribution:
  0 matches: 123,247
  1 matches: 119,157
  2 matches: 375,212
  3 matches: 530,841
  4 matches: 484,115
  5 matches: 321,957
  6 matches: 164,868
  7 matches: 63,968
  8 matches: 18,680
  9 matches: 4,205
  10 matches: 534
  11 matches: 37

Singletons:
  123,247
  Singleton rate: 5.58%

Matches by source:
  S2: 3,693,619
  S3: 3,944,746
  Unknown IDs: 0


# Classifying Ground Truth Matches by Source

In [18]:
from collections import Counter

def profile_match_sources(path, s2_ids, s3_ids, chunk_size=100_000):
    categories = Counter()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        chunksize=chunk_size
    ):
        for matched_ids in chunk["matched_entity_ids"].fillna(""):

            matched_ids = matched_ids.strip()

            # No matches
            if not matched_ids:
                categories["neither"] += 1
                continue

            ids = [
                entity_id.strip()
                for entity_id in matched_ids.split(",")
                if entity_id.strip()
            ]

            has_s2 = False
            has_s3 = False

            for entity_id in ids:
                if entity_id in s2_ids:
                    has_s2 = True
                elif entity_id in s3_ids:
                    has_s3 = True

            if has_s2 and has_s3:
                categories["both"] += 1
            elif has_s2:
                categories["s2_only"] += 1
            elif has_s3:
                categories["s3_only"] += 1
            else:
                categories["unknown"] += 1

    total = sum(categories.values())

    print(f"Total S1 businesses: {total:,}")

    print("\nMatch source categories:")

    for category in ["s2_only", "s3_only", "both", "neither", "unknown"]:
        count = categories[category]
        percentage = count / total * 100

        print(
            f"  {category:10} → "
            f"{count:>10,} ({percentage:.2f}%)"
        )

    return categories

In [19]:
source_categories = profile_match_sources(
    GT_PATH,
    s2_ids,
    s3_ids
)

Total S1 businesses: 2,206,821

Match source categories:
  s2_only    →    143,029 (6.48%)
  s3_only    →    164,498 (7.45%)
  both       →  1,776,047 (80.48%)
  neither    →    123,247 (5.58%)
  unknown    →          0 (0.00%)
